# Trabajo Practico 3 - Vision por Computadora

Detector final basado en SIFT + homografia para una instancia y detector unificado para multiples logos.
El notebook queda limpio y autocontenido.


In [ ]:
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np


def find_tp3_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / 'TP3',
        Path.cwd().parent / 'TP3',
        Path.cwd().parent.parent / 'TP3',
    ]
    for candidate in candidates:
        if (candidate / 'template' / 'pattern.png').exists():
            return candidate.resolve()
    return Path.cwd().resolve()


base_dir = find_tp3_dir()
images_dir = base_dir / 'images'
template_path = base_dir / 'template' / 'pattern.png'

image_paths = sorted([p.name for p in images_dir.iterdir() if p.suffix.lower() in {'.png', '.jpg'}])
images = {name: cv.imread(str(images_dir / name)) for name in image_paths}
images_rgb = {name: cv.cvtColor(image, cv.COLOR_BGR2RGB) for name, image in images.items() if image is not None}
images_gray = {name: cv.cvtColor(image, cv.COLOR_BGR2GRAY) for name, image in images.items() if image is not None}

template = cv.imread(str(template_path))
if template is None:
    raise FileNotFoundError(f'No se pudo cargar el template: {template_path}')
template_gray = cv.cvtColor(template, cv.COLOR_BGR2GRAY)
template_inv = cv.bitwise_not(template_gray)

sift = cv.SIFT_create()
flann_sift = cv.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50))
kp_sift_n, des_sift_n = sift.detectAndCompute(template_gray, None)
kp_sift_i, des_sift_i = sift.detectAndCompute(template_inv, None)


def match_and_locate(img_gray, kp_t, des_t, tpl_shape, detector, matcher, ratio=0.75):
    kp_i, des_i = detector.detectAndCompute(img_gray, None)
    if des_i is None or len(kp_i) < 2:
        return 0, None, [], kp_i, None

    matches = matcher.knnMatch(des_t, des_i, k=2)
    good = [m for m, n in matches if m.distance < ratio * n.distance]

    if len(good) < 4:
        return len(good), None, good, kp_i, None

    src = np.float32([kp_t[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp_i[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv.findHomography(src, dst, cv.RANSAC, 5.0)
    n_inliers = int(mask.sum()) if mask is not None else 0
    return n_inliers, H, good, kp_i, mask


def detect_logo_features(img_gray, detector, matcher, kp_n, des_n, kp_i, des_i, tpl_shape, ratio=0.75):
    h_t, w_t = tpl_shape[:2]
    corners = np.float32([[0, 0], [w_t, 0], [w_t, h_t], [0, h_t]]).reshape(-1, 1, 2)

    inl_n, H_n, good_n, _, _ = match_and_locate(img_gray, kp_n, des_n, tpl_shape, detector, matcher, ratio)
    inl_i, H_i, good_i, _, _ = match_and_locate(img_gray, kp_i, des_i, tpl_shape, detector, matcher, ratio)

    if inl_n >= inl_i and H_n is not None:
        quad = cv.perspectiveTransform(corners, H_n)
        return inl_n, quad, H_n, good_n
    if H_i is not None:
        quad = cv.perspectiveTransform(corners, H_i)
        return inl_i, quad, H_i, good_i
    return max(inl_n, inl_i), None, None, []


def draw_quad(img_rgb, quad, n_inliers, color=(0, 255, 0), thickness=3):
    img_draw = img_rgb.copy()
    if quad is not None:
        pts = quad.astype(np.int32)
        cv.polylines(img_draw, [pts], True, color, thickness)
        x, y = pts[0][0]
        cv.putText(img_draw, f'inliers={n_inliers}', (int(x), max(int(y) - 10, 20)), cv.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img_draw


def _quad_bbox(quad):
    pts = quad.reshape(-1, 2)
    mn, mx = pts.min(0), pts.max(0)
    return [float(mn[0]), float(mn[1]), float(mx[0] - mn[0]), float(mx[1] - mn[1])]


def _iou(box_a, box_b):
    x_a, y_a, w_a, h_a = box_a
    x_b, y_b, w_b, h_b = box_b
    x1, y1 = max(x_a, x_b), max(y_a, y_b)
    x2, y2 = min(x_a + w_a, x_b + w_b), min(y_a + h_a, y_b + h_b)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = w_a * h_a + w_b * h_b - inter
    return inter / union if union > 0 else 0.0


def _quad_valid(quad, image_shape, template_shape):
    h_t, w_t = template_shape[:2]
    h_img, w_img = image_shape[:2]
    pts = quad.reshape(-1, 2).astype(np.float32)
    x, y, w, h = _quad_bbox(quad)
    if w < 20 or h < 10:
        return False
    if w > w_img * 1.5 or h > h_img * 1.5:
        return False
    if cv.contourArea(pts) < 300:
        return False
    if not cv.isContourConvex(pts):
        return False
    r, r_t = w / h, w_t / h_t
    return 0.35 * r_t < r < 2.5 * r_t


def _fit_homography(cluster_members, votes, image_shape, template_shape, min_inliers):
    if len(cluster_members) < 4:
        return None

    h_t, w_t = template_shape[:2]
    src = np.float32([votes[i][2] for i in cluster_members]).reshape(-1, 1, 2)
    dst = np.float32([votes[i][1] for i in cluster_members]).reshape(-1, 1, 2)

    H, mask = cv.findHomography(src, dst, cv.RANSAC, 5.0)
    if H is None:
        return None

    n_inliers = int(mask.sum())
    if n_inliers < min_inliers:
        return None

    corners = np.float32([[0, 0], [w_t, 0], [w_t, h_t], [0, h_t]]).reshape(-1, 1, 2)
    quad = cv.perspectiveTransform(corners, H)

    if not _quad_valid(quad, image_shape, template_shape):
        return None
    return quad, n_inliers, _quad_bbox(quad)


def detect_logos_unified(img_gray, kp_normal, des_normal, kp_inverted, des_inverted, tpl_shape, detector, ratio=0.75, cell_size=40, min_votes=4, min_inliers=5, iou_thresh=0.3):
    h_t, w_t = tpl_shape[:2]
    tpl_center = np.array([w_t / 2, h_t / 2])
    kp_img, des_img = detector.detectAndCompute(img_gray, None)
    if des_img is None or len(kp_img) < 4:
        return []

    matcher = cv.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50))
    votes = []
    tpl_versions = [(kp_normal, des_normal), (kp_inverted, des_inverted)]

    for kp_tpl, des_tpl in tpl_versions:
        matches = matcher.knnMatch(des_img, des_tpl, k=2)

        for pair in matches:
            if len(pair) < 2:
                continue

            m, n = pair
            if m.distance >= ratio * n.distance:
                continue

            kp_i, kp_t = kp_img[m.queryIdx], kp_tpl[m.trainIdx]
            scale = kp_i.size / kp_t.size if kp_t.size > 0 else 1.0
            angle = np.deg2rad(kp_i.angle - kp_t.angle)
            c, s = np.cos(angle), np.sin(angle)

            v = tpl_center - np.array(kp_t.pt)
            v_rot = np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]]) * scale
            center = np.array(kp_i.pt) + v_rot

            votes.append((center, kp_i.pt, kp_t.pt))

    if len(votes) < min_inliers:
        return []

    acc = {}
    for i, (center, _, _) in enumerate(votes):
        key = (int(center[0] // cell_size), int(center[1] // cell_size))
        acc.setdefault(key, []).append(i)

    candidates = []
    for gx, gy in acc:
        cluster = []
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                cluster += acc.get((gx + dx, gy + dy), [])
        if len(cluster) < min_votes:
            continue
        res = _fit_homography(cluster, votes, img_gray.shape, tpl_shape, min_inliers)
        if res:
            candidates.append(res)

    res = _fit_homography(list(range(len(votes))), votes, img_gray.shape, tpl_shape, min_inliers)
    if res:
        candidates.append(res)

    candidates.sort(key=lambda c: c[1], reverse=True)

    final = []
    for quad, n_inliers, box in candidates:
        if any(_iou(box, saved_box) > iou_thresh for _, _, saved_box in final):
            continue
        final.append((quad, n_inliers, box))

    return [(quad, n_inliers) for quad, n_inliers, _ in final]


def draw_quads(img_rgb, instances, color=(0, 255, 0), thickness=2):
    img_draw = img_rgb.copy()
    for quad, n_inliers in instances:
        pts = quad.astype(np.int32)
        cv.polylines(img_draw, [pts], True, color, thickness)
        x, y = pts.reshape(-1, 2)[0]
        cv.putText(img_draw, str(n_inliers), (int(x), max(int(y) - 6, 14)), cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return img_draw


fig, axs = plt.subplots(2, 4, figsize=(20, 10))
axs = axs.flatten()
for i, name in enumerate(image_paths):
    img_gray = images_gray[name]
    n_inliers, quad, H, good = detect_logo_features(img_gray, sift, flann_sift, kp_sift_n, des_sift_n, kp_sift_i, des_sift_i, template_gray.shape)
    img_draw = draw_quad(images_rgb[name], quad, n_inliers)
    axs[i].imshow(img_draw)
    axs[i].set_title(f'{name}\ninliers={n_inliers}', fontsize=9)
    axs[i].axis('off')
for j in range(i + 1, len(axs)):
    axs[j].axis('off')
plt.suptitle('Item 1: Deteccion del logo con SIFT + homografia', fontsize=13)
plt.tight_layout()
plt.show()


detections_multi = detect_logos_unified(images_gray['coca_multi.png'], kp_sift_n, des_sift_n, kp_sift_i, des_sift_i, template_gray.shape, sift)
print(f'Instancias detectadas en coca_multi.png: {len(detections_multi)}')
for i, (quad, n_in) in enumerate(sorted(detections_multi, key=lambda d: -d[1])):
    x0, y0 = quad.reshape(-1, 2).min(0)
    print(f'  Deteccion {i+1}: inliers={n_in}, esquina sup-izq=({int(x0)},{int(y0)})')
img_draw = draw_quads(images_rgb['coca_multi.png'], detections_multi)
plt.figure(figsize=(14, 9))
plt.imshow(img_draw)
plt.title(f'Item 2: Detector unificado SIFT + Hough en coca_multi.png ({len(detections_multi)} logos)')
plt.axis('off')
plt.show()


fig, axs = plt.subplots(2, 4, figsize=(20, 10))
axs = axs.flatten()
for i, name in enumerate(image_paths):
    detections = detect_logos_unified(images_gray[name], kp_sift_n, des_sift_n, kp_sift_i, des_sift_i, template_gray.shape, sift)
    img_draw = draw_quads(images_rgb[name], detections)
    axs[i].imshow(img_draw)
    axs[i].set_title(f'{name}\n{len(detections)} det.', fontsize=9)
    axs[i].axis('off')
for j in range(i + 1, len(axs)):
    axs[j].axis('off')
plt.suptitle('Item 3: Detector unificado SIFT + Hough en todas las imagenes', fontsize=13)
plt.tight_layout()
plt.show()
